In [1]:
import torch
import torch.nn as nn
from torchvision import models

class EfficientNetModel(nn.Module):
    def __init__(self, num_classes):
        super(EfficientNetModel, self).__init__()
        self.model = models.efficientnet_b0(pretrained=False)  # IMPORTANT: pretrained=False when loading weights
        self.model.classifier[1] = nn.Linear(
            self.model.classifier[1].in_features,
            num_classes
        )

    def forward(self, x):
        return self.model(x)


In [2]:
import torch

# Load your model
checkpoint = torch.load('EfficientNet_best_78.21%.pth', map_location='cpu')

# Get the state_dict
if 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
else:
    state_dict = checkpoint

# Print all layers with their shapes
for name, param in state_dict.items():
    print(f"{name:30s} → {param.shape}")


## 📊 Read the Output:

# You'll see something like:
# ```
# fc1.weight                     → torch.Size([256, 128])
# fc1.bias                       → torch.Size([256])
# fc2.weight                     → torch.Size([128, 256])
# fc3.weight                     → torch.Size([5, 128])

model.features.0.0.weight      → torch.Size([32, 3, 3, 3])
model.features.0.1.weight      → torch.Size([32])
model.features.0.1.bias        → torch.Size([32])
model.features.0.1.running_mean → torch.Size([32])
model.features.0.1.running_var → torch.Size([32])
model.features.0.1.num_batches_tracked → torch.Size([])
model.features.1.0.block.0.0.weight → torch.Size([32, 1, 3, 3])
model.features.1.0.block.0.1.weight → torch.Size([32])
model.features.1.0.block.0.1.bias → torch.Size([32])
model.features.1.0.block.0.1.running_mean → torch.Size([32])
model.features.1.0.block.0.1.running_var → torch.Size([32])
model.features.1.0.block.0.1.num_batches_tracked → torch.Size([])
model.features.1.0.block.1.fc1.weight → torch.Size([8, 32, 1, 1])
model.features.1.0.block.1.fc1.bias → torch.Size([8])
model.features.1.0.block.1.fc2.weight → torch.Size([32, 8, 1, 1])
model.features.1.0.block.1.fc2.bias → torch.Size([32])
model.features.1.0.block.2.0.weight → torch.Size([16, 32, 1, 1])
model.features.1.0.

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = 4  # MUST match training
model = EfficientNetModel(num_classes).to(device)

model.load_state_dict(torch.load("EfficientNet_best_78.21%.pth", map_location=device))

model.eval()  # VERY IMPORTANT


EfficientNetModel(
  (model): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(inplace=True)
              (sca

In [10]:
CLASS_NAMES = ['kaiyuan', 'noise', 'speedboat', 'uuv']

In [11]:
#single image testing 
from PIL import Image
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

image = Image.open(r"C:\Users\yusle\OneDrive\Desktop\Git Hub Repositories\OSINModels2\classes\2_spec\chunk_0016.png").convert("RGB")
image = transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    outputs = model(image)
    prediction = torch.argmax(outputs, dim=1)

import torch.nn.functional as F

with torch.no_grad():
    outputs = model(image)
    probs = F.softmax(outputs, dim=1)
    print("Probabilities:", probs)

# print(image.mean().item()) #for testing if the images are the same 
print("Predicted class:", CLASS_NAMES[prediction.item()])


Probabilities: tensor([[6.8704e-12, 1.0000e+00, 8.4387e-18, 3.5175e-09]], device='cuda:0')
Predicted class: noise


In [33]:
test_folder=r"C:\Users\yusle\OneDrive\Desktop\Git Hub Repositories\OSINModels2\classes\1_spec"

In [34]:
import os
#looping testing 
for file in os.listdir(test_folder): 
    file_path= os.path.join(test_folder,file)
    # print(file_path)

    image = Image.open(file_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        prediction = torch.argmax(outputs, dim=1)

    print("Predicted class:", prediction.item())


Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 3
Predicted class: 2
Predicted class: 2
Predicted class: 3
Predicted class: 3
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 3
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 3
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 3
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 3
Predicted class: 3
Predicted class: 2
Predicted class: 3
Predicted class: 0
Predicted class: 2
Predicted class: 2
Predicted class: 3
Predicted class: 3
Predicted class: 3
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 2
Predicted class: 3
Predicted class: 2
Predicted class: 3
Predicted class: 3
Predicted class: 2
Predicted class: 2
Predicted cl

KeyboardInterrupt: 